# Task 1: News Topic Classifier Using BERT

**Objective:** Fine-tune a BERT transformer model to classify news headlines into topic categories using the AG News dataset.

**Skills:** NLP, Transfer Learning, Fine-Tuning, Evaluation Metrics, Model Deployment

## Install Dependencies

In [ ]:
# Install required libraries
!pip install -q transformers datasets torch scikit-learn gradio

print("✅ All libraries installed successfully!")

## Import Libraries & Setup

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Libraries loaded! Using device: {device}")

## Load the AG News Dataset

In [ ]:
# Loading AG News from Hugging Face Datasets
# It has 4 categories: World, Sports, Business, Sci/Tech
print("📥 Loading AG News dataset from Hugging Face...")

dataset = load_dataset("ag_news")

# AG News label mapping
label_names = ["World", "Sports", "Business", "Sci/Tech"]

print(f"Train samples : {len(dataset['train'])}")
print(f"Test samples  : {len(dataset['test'])}")
print("\nSample entry:")
print(dataset['train'][0])
print(f"\nLabels: {label_names}")

## Explore & Visualize the Dataset

In [ ]:
# Check class distribution in training set
train_df = pd.DataFrame(dataset['train'])
train_df['label_name'] = train_df['label'].map(dict(enumerate(label_names)))

plt.figure(figsize=(7, 4))
ax = train_df['label_name'].value_counts().plot(kind='bar', color=['#2196F3','#4CAF50','#FF9800','#9C27B0'], edgecolor='white')
plt.title('AG News – Class Distribution (Training Set)', fontsize=13, pad=12)
plt.xlabel('News Category', fontsize=11)
plt.ylabel('Number of Samples', fontsize=11)
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 Class counts:")
print(train_df['label_name'].value_counts())

## Tokenize the Dataset

In [ ]:
# Load BERT tokenizer
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

# Subsample for faster training in Colab (use full set for production)
# Using 4000 train + 1000 test for quick experimentation
from datasets import DatasetDict
small_dataset = DatasetDict({
    "train": dataset["train"].shuffle(seed=42).select(range(4000)),
    "test":  dataset["test"].shuffle(seed=42).select(range(1000))
})

def tokenize_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

print("⚙️ Tokenizing dataset...")
tokenized = small_dataset.map(tokenize_fn, batched=True)
tokenized = tokenized.rename_column("label", "labels")
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print("✅ Tokenization complete!")
print(f"Train size : {len(tokenized['train'])}")
print(f"Test size  : {len(tokenized['test'])}")

## Load & Fine-Tune BERT

In [ ]:
# Load pre-trained BERT model for sequence classification (4 output classes)
print("🤖 Loading bert-base-uncased model...")
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)
model.to(device)
print("✅ Model loaded and moved to device!")

# Define training arguments
training_args = TrainingArguments(
    output_dir="./bert_ag_news",
    num_train_epochs=2,             # 2 epochs is enough for demo; use 3-5 for full training
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none",               # Disable W&B for simplicity
)

# Custom compute_metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    f1  = f1_score(labels, preds, average="weighted")
    return {"accuracy": acc, "f1": f1}

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    compute_metrics=compute_metrics,
)

print("\n🚀 Starting fine-tuning...")
trainer.train()
print("\n✅ Fine-tuning complete!")

## Evaluate the Model

In [ ]:
# Run evaluation on test set
print("📊 Evaluating model on test set...")
results = trainer.evaluate()
print(f"\nAccuracy : {results['eval_accuracy']:.4f}")
print(f"F1-Score  : {results['eval_f1']:.4f}")

# Detailed classification report
preds_output = trainer.predict(tokenized["test"])
y_pred = np.argmax(preds_output.predictions, axis=-1)
y_true = preds_output.label_ids

print("\n📋 Detailed Classification Report:")
print(classification_report(y_true, y_pred, target_names=label_names))

## Confusion Matrix

In [ ]:
# Visualize confusion matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names)
plt.title('Confusion Matrix – BERT AG News Classifier', fontsize=13, pad=12)
plt.xlabel('Predicted Label', fontsize=11)
plt.ylabel('True Label', fontsize=11)
plt.tight_layout()
plt.show()

## Save the Model

In [ ]:
# Save fine-tuned model and tokenizer for deployment
model.save_pretrained("./bert_ag_news_final")
tokenizer.save_pretrained("./bert_ag_news_final")
print("💾 Model and tokenizer saved to ./bert_ag_news_final")

## Deploy with Gradio

In [ ]:
import gradio as gr

# Load saved model for inference
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="./bert_ag_news_final",
    tokenizer="./bert_ag_news_final",
    device=0 if torch.cuda.is_available() else -1
)

label_map = {"LABEL_0": "🌍 World", "LABEL_1": "⚽ Sports",
             "LABEL_2": "💼 Business", "LABEL_3": "🔬 Sci/Tech"}

def predict_category(text):
    result = classifier(text, truncation=True, max_length=128)[0]
    label = label_map.get(result["label"], result["label"])
    confidence = f"{result['score']*100:.1f}%"
    return f"Category: {label}\nConfidence: {confidence}"

# Launch Gradio demo
demo = gr.Interface(
    fn=predict_category,
    inputs=gr.Textbox(lines=3, placeholder="Paste a news headline here...", label="News Headline"),
    outputs=gr.Textbox(label="Prediction"),
    title="📰 News Topic Classifier (BERT)",
    description="Fine-tuned BERT model classifying AG News into: World | Sports | Business | Sci/Tech",
    examples=[
        ["Apple launches new iPhone with groundbreaking AI features"],
        ["Manchester United defeats Arsenal in a thrilling 3-2 comeback"],
        ["Federal Reserve raises interest rates amid inflation concerns"],
        ["Scientists discover new exoplanet with potential for life"],
    ]
)

demo.launch(share=True)  # share=True gives a public URL in Colab

## Final Summary & Insights

In [ ]:
print("=" * 65)
print("🎉 TASK 1 COMPLETED – NEWS TOPIC CLASSIFIER USING BERT")
print("=" * 65)
print(f"  Model      : bert-base-uncased (fine-tuned)")
print(f"  Dataset    : AG News (4 categories)")
print(f"  Accuracy   : {results['eval_accuracy']:.4f}")
print(f"  F1-Score   : {results['eval_f1']:.4f}")
print()
print("Key Insights:")
print("  • BERT's pre-trained language understanding transfers")
print("    very well to news classification with just 2 epochs.")
print("  • Weighted F1 handles any class imbalance gracefully.")
print("  • The Gradio app allows live, interactive demo of the model.")
print("  • For production: train on the full dataset (120k samples)")
print("    and increase epochs to 3-5 for higher accuracy.")
print("=" * 65)